In [17]:
import torch
import jax

In [18]:
print(torch.xpu.is_available())

True


In [19]:
print(jax.devices())

INFO: Intel Extension for OpenXLA version: 0.6.0, commit: 655abd7e
ERROR:2025-08-19 18:28:39,403:jax._src.xla_bridge:502: Jax plugin configuration error: Exception when calling jax_plugins.intel_extension_for_openxla.initialize()
Traceback (most recent call last):
  File "/home/kivalm/code/AIFNIPD/.venv/lib/python3.13/site-packages/jax/_src/xla_bridge.py", line 500, in discover_pjrt_plugins
    plugin_module.initialize()
    ~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "/home/kivalm/code/AIFNIPD/.venv/lib/python3.13/site-packages/jax_plugins/intel_extension_for_openxla/__init__.py", line 58, in initialize
    c_api = xb.register_plugin("sycl",
                       priority=500,
                       library_path=str(path),
                       options=options)
  File "/home/kivalm/code/AIFNIPD/.venv/lib/python3.13/site-packages/jax/_src/xla_bridge.py", line 629, in register_plugin
    c_api = xla_client.load_pjrt_plugin_dynamically(plugin_name, library_path)
  File "/home/kivalm/code/AIFNIPD

[CpuDevice(id=0)]


In [13]:
device = "cpu"

In [14]:
import torch
import torch.nn as nn

# Simple dataset: numbers 0-9, label 0 for even, 1 for odd
X = torch.arange(0, 10, dtype=torch.float32).unsqueeze(1)
X = X.to(device)
y = (X % 2).long().squeeze(1)
y = y.to(device)

# Simple model: 1 input, 1 hidden, 2 output (even/odd)
class EvenOddClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 8),
            nn.ReLU(),
            nn.Linear(8, 2)
        )
    def forward(self, x):
        return self.net(x)

model = EvenOddClassifier()
model.to(device)
print(model)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)


EvenOddClassifier(
  (net): Sequential(
    (0): Linear(in_features=1, out_features=8, bias=True)
    (1): ReLU()
    (2): Linear(in_features=8, out_features=2, bias=True)
  )
)


In [ ]:
# Training loop
import tqdm
for epoch in range(100000):
    optimizer.zero_grad()
    outputs = model(X)
    loss = criterion(outputs, y)
    loss.backward()
    optimizer.step()

In [16]:
# Test
with torch.no_grad():
    test_X = torch.arange(0, 10, dtype=torch.float32).unsqueeze(1)
    test_X = test_X.to("xpu")
    preds = model(test_X).argmax(dim=1)
    print("Predictions:", preds.tolist())
    print("Ground truth:", y.tolist())

RuntimeError: Expected all tensors to be on the same device, but got mat1 is on xpu:0, different from other tensors on cpu (when checking argument in method wrapper_XPU_addmm)